# Quick guide: differentiating a QG trajectory

The model is differentiable: any function of the solution can be
differentiated with respect to its inputs. Whether that is practical is
decided by two quantities, measured below:

- the cost of the backward pass relative to the forward run — **~2.5x**
  with the default settings (V100, 400 steps);
- the memory it needs — with the default settings, independent of
  trajectory length; without checkpointing, memory grows with the
  number of steps and the backward pass no longer fits on long
  trajectories.

Both are controlled by `checkpoint_rate` in
`Model.generate_integration_fn`. Checkpointing (`jax.checkpoint`) stores
the state every *k* steps instead of every intermediate and recomputes
the missing intermediates from the nearest stored state during the
backward pass; *k* is `checkpoint_rate`. It changes memory and runtime
only; the gradient is identical for every setting. When
`checkpoint_rate > 1`, `unroll` (clamped to `checkpoint_rate`) controls
how far the compiler unrolls the loop within one checkpoint interval.


In [1]:
import time

import numpy as np

import jax
import jax.numpy as jnp

from jaxqg import create_two_layer_model, define_state
from jaxqg.utils import gaussian_random_field

print("jax", jax.__version__, "| backend:", jax.default_backend(),
      "| x64:", jax.config.jax_enable_x64)


def build_benchmark(nx):
    """Two-layer model plus a matched spectral-slope initial condition."""
    model = create_two_layer_model(n_x=nx, U_1=0.0, U_2=0.0,
                                    beta=1e-11, gamma=1e-7, dt=1800.0)
    qh0 = gaussian_random_field(model, 1.2e-13 * model.grid.wv**-2)
    return model, qh0.at[:, model.grid.wv2_nd < 4].set(0.0)


def bench(fn, reps=3):
    """Mean wall-clock time of fn, excluding the first (warm-up) call."""
    out = fn()
    jax.block_until_ready(out)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        out = fn()
        jax.block_until_ready(out)
        ts.append(time.perf_counter() - t0)
    return float(np.mean(ts))

jax 0.11.2 | backend: gpu | x64: True


## The gradient is independent of the settings

Checkpointing recomputes rather than stores intermediates. It does not
change the gradient. The cell below verifies this at smoke scale
(seconds on any backend), for an example objective: the final-state PV
norm differentiated with respect to the initial condition.

In [2]:
N_STEPS = 100
model, qh0 = build_benchmark(128)


def make_grad(model, n_steps, checkpoint_rate, unroll=1):
    """value_and_grad of the final-state loss at a given checkpoint_rate."""
    integrate = model.generate_integration_fn(
        n_steps, sample_rate=n_steps, return_final_state=True,
        checkpoint_rate=checkpoint_rate, unroll=unroll)

    def loss(qh):
        final_state, _ = integrate(define_state(model, qh))
        return jnp.real(jnp.mean(jnp.abs(final_state.qh) ** 2))

    return jax.jit(jax.value_and_grad(loss))


def make_plain_grad(model, n_steps):
    """Un-checkpointed reference: a bare fori_loop, every intermediate kept."""
    def loss(qh):
        state = jax.lax.fori_loop(
            0, n_steps, model.evolve_one_step_stateless, define_state(model, qh))
        return jnp.real(jnp.mean(jnp.abs(state.qh) ** 2))

    return jax.jit(jax.value_and_grad(loss))


grads = {
    "plain": make_plain_grad(model, N_STEPS)(qh0)[1],
    "rate=1 (default)": make_grad(model, N_STEPS, 1)(qh0)[1],
    "rate=4": make_grad(model, N_STEPS, 4)(qh0)[1],
}
for g in grads.values():
    jax.block_until_ready(g)
ref = grads["rate=1 (default)"]
for name, g in grads.items():
    rel = float(jnp.max(jnp.abs(g - ref)) / jnp.max(jnp.abs(ref)))
    print(f"grad rel diff  {name:<18} {rel:.0e}")

grad rel diff  plain              0e+00
grad rel diff  rate=1 (default)   0e+00
grad rel diff  rate=4             0e+00


## Forward versus backward cost

Timing at full scale: a 400-step trajectory at 512². The cell times the
forward integration and the backward pass, with and without
checkpointing. Compilation dominates the wall clock on the first run;
the timed calls themselves take seconds.


In [3]:
# 512^2 grid, 400 steps. Compilation takes a few minutes on GPU.
N_STEPS = 400
model, qh0 = build_benchmark(512)
state0 = define_state(model, qh0)

forward = jax.jit(model.generate_integration_fn(
    N_STEPS, sample_rate=N_STEPS, return_final_state=True))
t_fwd = bench(lambda: forward(state0)[0])

results = {"forward": t_fwd}
for name, vg in {
    "plain AD": make_plain_grad(model, N_STEPS),
    "ckpt rate=1 (default)": make_grad(model, N_STEPS, 1),
}.items():
    try:
        _, g = vg(qh0)
        jax.block_until_ready(g)
        results[name] = bench(lambda: vg(qh0)[0])
    except Exception as exc:
        print(f"{name}: {type(exc).__name__} — tape does not fit in device memory")

print(f"\n{model.grid.n_x}^2 grid, {N_STEPS} steps, backend={jax.default_backend()}")
for name, t in results.items():
    print(f"  {name:<22} {t:7.2f} s" +
          ("" if name == "forward" else f"  = {t / t_fwd:4.1f}x forward"))


512^2 grid, 400 steps, backend=gpu
  forward                   0.27 s
  plain AD                  0.63 s  =  2.3x forward
  ckpt rate=1 (default)     0.69 s  =  2.5x forward


## Reading the numbers

- The checkpointed backward pass costs a small multiple of the forward
  run (~2.5x here). The total is the sum of three contributions — the
  forward replay, the recompute of each checkpoint interval, and the
  adjoint — which each scale with the forward sweep, so the ratio is
  insensitive to trajectory length.
- Plain AD is cheaper when it fits, but its memory grows with the
  number of steps: this trajectory fits in device memory, a few times
  longer would not. The checkpointed run holds a single timestep's
  intermediates, so it is unaffected by trajectory length.
- Use the defaults on GPU (`checkpoint_rate=1`, `unroll=1`). On
  CPU/Metal, where per-op dispatch makes checkpoint boundaries
  expensive, `checkpoint_rate=8, unroll=8` is faster.

`sample_rate`, which selects which states the forward run returns,
plays no role here.
